# Simulation, stress and walk-forward backtesting

The three engines that answer "what could happen" and "would this process have
worked". Same pattern as the first notebook: the engine driven directly, with no
server involved.

In [ ]:
import sys
from pathlib import Path

# The engine is a plain package at the repository root: no server, no database.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("ready")

In [ ]:
from quant.data.providers import fetch_prices
from quant.data.validation import to_price_matrix, validate_prices
from quant.features.returns import simple_returns
from quant.optimization.estimators import estimate_covariance, estimate_expected_returns

SOURCE = "synthetic"
TICKERS = ["AAPL", "MSFT", "NVDA", "JPM", "XOM", "SPY", "TLT", "GLD"]

clean, _ = validate_prices(fetch_prices(SOURCE, TICKERS, "2019-01-02", None))
prices = to_price_matrix(clean)[TICKERS]
rets = simple_returns(prices)

w = pd.Series({"AAPL": 0.18, "MSFT": 0.18, "NVDA": 0.12, "JPM": 0.12,
               "XOM": 0.10, "SPY": 0.15, "TLT": 0.10, "GLD": 0.05}).reindex(TICKERS).fillna(0.0)
mu = estimate_expected_returns(rets)
cov = estimate_covariance(rets, method="ledoit_wolf")
CAPITAL = 100_000.0
print(f"{len(rets)} observations to {rets.index[-1].date()}")

## 1. Monte Carlo

Each simulation draws a sequence of daily portfolio returns and compounds them
from today's value. Compare the three distributions: the normal one is the
optimistic reading of the tail.

In [ ]:
from quant.simulation.monte_carlo import run_monte_carlo

results = {}
for method in ("normal", "student_t", "bootstrap"):
    results[method] = run_monte_carlo(
        w, mu, cov, initial_value=CAPITAL, horizon_days=252, n_sims=4000,
        method=method, historical_returns=rets, seed=42)

pd.DataFrame({
    m: {"median": r["terminal"]["median"], "5th pct": r["terminal"]["p5"],
        "1st pct": r["terminal"]["p1"], "P(loss)": r["prob_loss"],
        "P(loss > 10%)": r["prob_loss_threshold"], "VaR 95%": r["var_95"], "ES 95%": r["es_95"]}
    for m, r in results.items()}).T

Same expected return and the same covariance in every row; only the shape of
the distribution changes. The 1st percentile moving by a wide margin between
`normal` and `student_t` is the whole argument against reading a normal-model
tail as if it were the real one.

In [ ]:
fig, ax = plt.subplots()
for method, r in results.items():
    bands = pd.DataFrame(r["bands"])
    ax.plot(bands["day"], bands["p50"], lw=2, label=f"{method} median")
    ax.fill_between(bands["day"], bands["p5"], bands["p95"], alpha=0.12)
ax.axhline(CAPITAL, ls="--", lw=1, color="gray")
ax.set_xlabel("trading days ahead")
ax.set_ylabel("portfolio value")
ax.set_title("Median path and the 5th-95th percentile band, by distribution")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. Stress testing

Not a probability statement: one chosen adverse assumption, applied to current
weights, with the loss attributed per holding.

In [ ]:
from quant.data.universe import asset_metadata
from quant.factors.market import asset_betas
from quant.stress.scenarios import PREDEFINED_SCENARIOS, run_stress_test

meta = asset_metadata()
betas = asset_betas(rets, rets["SPY"])["beta"].to_dict()
weights = {t: float(x) for t, x in w.items() if x > 0}

rows = []
for sc in PREDEFINED_SCENARIOS:
    r = run_stress_test(sc, weights, CAPITAL, meta, betas, prices)
    worst = r["largest_contributors"][0] if r["largest_contributors"] else {"ticker": "-", "pnl": 0.0}
    rows.append({"scenario": sc.name, "change": r["pct_change"], "pnl": r["pnl"],
                 "worst position": worst["ticker"], "its pnl": worst["pnl"]})

pd.DataFrame(rows).sort_values("change")

## 3. Walk-forward backtest

At each rebalance date the optimizer sees a window ending **strictly before**
that date, picks weights, pays costs on the turnover, and then holds them while
they drift with prices. No strategy can see the day it trades on.

In [ ]:
from quant.backtest.walk_forward import walk_forward_backtest
from quant.optimization.optimizer import Constraints

bt = walk_forward_backtest(
    prices, ["equal_weight", "min_variance", "max_sharpe", "risk_parity"],
    benchmark_prices=prices["SPY"], lookback=252, frequency="quarterly",
    cost_rate=0.001, constraints=Constraints(max_weight=0.35),
    risk_free_rate=0.04, sectors={}, initial_capital=CAPITAL)

print(bt["integrity"])
pd.DataFrame(bt["metrics"]).T[["total_return", "cagr", "volatility", "sharpe",
                               "max_drawdown", "avg_turnover", "total_costs"]]

In [ ]:
series = pd.DataFrame(bt["series"]).set_index("date")
value_cols = [c for c in series.columns if not c.endswith("_dd")]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})
series[value_cols].plot(ax=ax1, lw=2)
ax1.set_title(f"Walk-forward {bt['start']} to {bt['end']}, quarterly, 10 bps per trade")
ax1.set_ylabel("portfolio value")
ax1.legend(fontsize=8)

series[[f"{c}_dd" for c in value_cols]].plot(ax=ax2, lw=1.5, legend=False)
ax2.set_ylabel("drawdown")
ax2.set_xlabel("")
plt.tight_layout()
plt.show()

### The no-look-ahead check, verified rather than asserted

Every rebalance logs the window it used. That window must end before the date it
traded on, with no exceptions.

In [ ]:
bad = []
for strat, log in bt["rebalances"].items():
    for entry in log:
        if "window_end" in entry and not entry["window_end"] < entry["date"]:
            bad.append((strat, entry["date"], entry["window_end"]))

print("look-ahead violations:", bad or "none")
pd.DataFrame(bt["rebalances"]["max_sharpe"])[["date", "window_start", "window_end", "turnover", "cost"]].head(8)

### How sensitive is the ranking?

One historical path is one sample. Change the frequency and the window, and see
whether the winner survives.

In [ ]:
rows = []
for freq in ("monthly", "quarterly", "annual"):
    for lookback in (126, 252, 504):
        r = walk_forward_backtest(
            prices, ["equal_weight", "min_variance", "max_sharpe", "risk_parity"],
            benchmark_prices=prices["SPY"], lookback=lookback, frequency=freq,
            cost_rate=0.001, constraints=Constraints(max_weight=0.35),
            risk_free_rate=0.04, sectors={}, initial_capital=CAPITAL)
        best = max((k for k in r["metrics"] if k != "benchmark"),
                   key=lambda k: r["metrics"][k]["sharpe"] or -99)
        rows.append({"frequency": freq, "lookback": lookback, "best by Sharpe": best,
                     "its sharpe": r["metrics"][best]["sharpe"],
                     "equal weight sharpe": r["metrics"]["equal_weight"]["sharpe"]})

pd.DataFrame(rows)

If the winner changes from row to row, the backtest is measuring noise rather
than an edge — and equal weight holding up across most of the grid is the usual
and honest outcome. Worth knowing before putting money behind an optimizer.